<a href="https://colab.research.google.com/github/giannibellini63/ipynb_projects/blob/main/analizzatore_filesystem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analizzatore File System con Python
Mini-progetto per confrontare sistemi operativi e file system.


In [ ]:
import os
import platform
import time
import stat

**Scansione folder**

In [ ]:
import os

for root, dirs, files in os.walk("."):
    for f in files:
      filename = os.path.join(root, f)
      print(filename)

**Stat**

In [ ]:
import os

full_path = "sample_data/README.md"
print(os.stat(full_path))

os.stat_result(st_mode=33261, st_ino=1179656, st_dev=52, st_nlink=1, st_uid=0, st_gid=0, st_size=962, st_atime=946713600, st_mtime=946713600, st_ctime=1773772232)


**Dettaglio Stat**

In [ ]:
import os

full_path = "sample_data/README.md"
print(os.stat(full_path))
print(os.stat(full_path).st_size)

In [ ]:
def get_os_info():
    return {
        'os_name': os.name,
        'platform': platform.system(),
        'release': platform.release()
    }

In [ ]:
def get_permissions(info):
    mode = info.st_mode
    return {
        'readable': bool(mode & stat.S_IRUSR),
        'writable': bool(mode & stat.S_IWUSR),
        'executable': bool(mode & stat.S_IXUSR),
        'raw': oct(mode)
    }

In [ ]:



def analyze_directory(path):
    total_files = 0
    total_dirs = 0
    total_size = 0
    largest_file = ('', 0)
    results = []

    for root, dirs, files in os.walk(path):
        total_dirs += len(dirs)

        for name in files:
            total_files += 1
            full_path = os.path.join(root, name)

            try:
                info = os.stat(full_path)
                size = info.st_size
                total_size += size

                if size > largest_file[1]:
                    largest_file = (full_path, size)

                results.append({
                    'path': full_path,
                    'size': size,
                    'last_modified': time.ctime(info.st_mtime),
                    'permissions': get_permissions(info)
                })

            except Exception as e:
                print(f'Errore su {full_path}: {e}')

    return {
        'total_files': total_files,
        'total_dirs': total_dirs,
        'total_size': total_size,
        'largest_file': largest_file,
        'details': results
    }

def format_size(size):
    for unit in ['B','KB','MB','GB']:
        if size < 1024:
            return f"{size:.2f} {unit}"
        size /= 1024

def print_report(os_info, analysis):
    print('=== SISTEMA OPERATIVO ===')
    print(os_info)

    print('\n=== STATISTICHE ===')
    print('File:', analysis['total_files'])
    print('Directory:', analysis['total_dirs'])
    print('Dimensione totale:', format_size(analysis['total_size']))

    largest = analysis['largest_file']
    print('\n=== FILE PIÙ GRANDE ===')
    print(largest[0], format_size(largest[1]))

    print('\n=== ESEMPIO FILE ===')
    for file in analysis['details'][:5]:
        print(file)


In [ ]:
path = '.'  # modifica qui se vuoi analizzare un'altra cartella
os_info = get_os_info()
analysis = analyze_directory(path)
print_report(os_info, analysis)
